## **Task 2: Hybrid Chatbot for FAQs (Final Implementation)**

The FAQ Chatbot was implemented as a robust hybrid system, combining local document retrieval with a powerful Generative AI fallback to ensure a relevant answer for every query.

Implementation Summary:
Data and Preprocessing: A set of Python programming FAQs was established. The nltk library was used to clean the text via tokenization, stop-word removal, and lemmatization, preparing both the FAQ and user input for vectorization.

Retrieval Logic (TF-IDF & Cosine Similarity):
The TfidfVectorizer from scikit-learn was trained on the preprocessed FAQ questions to convert them into numerical vectors.
The user's query was transformed into a vector, and the cosine_similarity function was used to calculate the similarity score between the query and all FAQ vectors.

Hybrid Logic (Generative Fallback): A high SIMILARITY_THRESHOLD was set.
Retrieval Path: If the similarity score was above the threshold, the most relevant, pre-written FAQ answer was returned.

Generative Path: If the score was below the threshold (meaning the question was new or too broad), the system defaulted to the Gemini API to generate a precise, on-topic answer, ensuring no question was left unanswered.

Professional UI: The UI was enhanced using the colorama library to create a highly professional, formatted, and color-coded command-line interface, clearly distinguishing between user input, Retrieval answers, and Generative (Gemini AI) answers.

### **Installing Dependencies**

In [4]:
!pip install google-genai colorama nltk scikit-learn -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [12]:
# --- 0. Setup and Imports
import nltk
import string
import warnings
import os
import textwrap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai.errors import APIError
from colorama import Fore, Style, init

init(autoreset=True)
warnings.filterwarnings('ignore')

# --- 1. DATA & CONFIGURATION ---
faq_data = {
    # Keep the specific, technical FAQs
    "How to install Python?":
        "The recommended way to install Python is to download the latest stable version from the official website (python.org). It's crucial to check the 'Add Python to PATH' option during the Windows installation process for easy command-line access. For Mac/Linux, it's often pre-installed or easily added via package managers (like apt or brew).",
    "What is pip?":
        "pip is the standard package manager for Python. It allows users to install and manage software packages and dependencies from the Python Package Index (PyPI). To install a package, you simply run 'pip install [package_name]' in your terminal.",
    "Explain the GIL.":
        "The Global Interpreter Lock (GIL) is a mutex (mutual exclusion lock) that protects access to Python objects, preventing multiple native threads from executing Python bytecodes simultaneously. While it simplifies memory management, it effectively means Python threads cannot use multiple CPU cores for CPU-bound tasks.",
    "What are the best Python libraries for Machine Learning?":
        "For Machine Learning, the core libraries are: **NumPy** (numerical operations), **Pandas** (data manipulation), and **Scikit-learn** (standard ML algorithms). For Deep Learning, the leading frameworks are **TensorFlow** (and Keras) and **PyTorch**.",
    "What is the difference between tuple and list?":
        "The main difference is mutability: **Lists** are *mutable* (can be changed, added to, or removed from), while **Tuples** are *immutable* (cannot be changed after creation). Lists are defined by `[]` and tuples by `()`. Tuples are generally faster and safer for fixed data."
}

faq_questions = list(faq_data.keys())
faq_answers = list(faq_data.values())

# NLP components
lemmatizer = nltk.stem.WordNetLemmatizer()
stop_words = nltk.corpus.stopwords.words('english')
PUNCTUATION = string.punctuation
SIMILARITY_THRESHOLD = 0.75 # Increased threshold to ensure high-confidence, specific matches only

# --- 2. TEXT PREPROCESSING & TF-IDF
def preprocess_text(text):
    tokens = nltk.word_tokenize(text.lower())
    tokens = [word for word in tokens if word not in PUNCTUATION and word.isalnum()]
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

processed_faq_questions = [preprocess_text(q) for q in faq_questions]
vectorizer = TfidfVectorizer()
faq_vectors = vectorizer.fit_transform(processed_faq_questions)


# --- 3. GENERATIVE FALLBACK (GEMINI API)
try:
    client = genai.Client()
except Exception:
    client = None

def generate_gemini_response(prompt):
    if not client:
        return f"Gemini API is not configured. Cannot generate answers for new questions. {Fore.RED}Please set the API Key.{Style.RESET_ALL}"

    system_instruction = (
        "You are an expert Python programming assistant. Your goal is to provide concise, "
        "accurate, and professional answers about the Python language, its libraries, and "
        "related development concepts. Only answer questions related to the Python ecosystem."
    )

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={'system_instruction': system_instruction}
        )
        return response.text.replace('**', Style.BRIGHT).replace('*', Style.RESET_ALL)
    except APIError as e:
        return f"A Gemini API error occurred: Could not generate content. ({e})"
    except Exception as e:
        return f"An unknown error occurred during API call. ({e})"


# --- 4. HYBRID CHATBOT LOGIC

def hybrid_chatbot_response(user_input):
    processed_input = preprocess_text(user_input)

    if processed_input:
        user_vector = vectorizer.transform([processed_input])
        similarity_scores = cosine_similarity(user_vector, faq_vectors)
        best_match_index = similarity_scores.argmax()
        max_similarity = similarity_scores[0, best_match_index]

        if max_similarity >= SIMILARITY_THRESHOLD:
            return faq_answers[best_match_index], max_similarity, "RETRIEVAL"

    # Generative Fallback is now the default for broad/new questions
    return generate_gemini_response(user_input), 0.999, "GENERATIVE"


# --- 5. CHAT UI

def format_bot_response(response, similarity, mode):

    if mode == "RETRIEVAL":
        status_color = Fore.YELLOW + Style.BRIGHT
        mode_label = f"FAQ Match: {similarity*100:.1f}%"
    elif mode == "GENERATIVE":
        status_color = Fore.GREEN + Style.BRIGHT
        mode_label = "Gemini AI"
    else:
        status_color = Fore.WHITE + Style.BRIGHT
        mode_label = "System"

    header_line = f" 🤖 BOT Response ({mode_label}):"
    wrapped_response = textwrap.fill(response, width=78, subsequent_indent='   ')

    output = f"\n{Fore.CYAN}╔{'═' * 80}╗"
    output += f"\n{Fore.CYAN}║ {Fore.MAGENTA}{header_line}{status_color}{' ' * (80 - len(header_line) - 1 - len(status_color))} ║"
    output += f"\n{Fore.CYAN}╠{'─' * 80}╣"
    output += f"\n{Fore.CYAN}║ {Fore.RESET}{wrapped_response.replace('\n', f'\n{Fore.CYAN}║ {Fore.RESET}')}"
    last_line_len = len(wrapped_response.split('\n')[-1])
    padding = max(0, 79 - last_line_len)
    output += f"{Fore.CYAN}{' ' * padding}║"
    output += f"\n{Fore.CYAN}╚{'═' * 80}╝{Style.RESET_ALL}\n"

    return output


def start_chat():

    print(f"\n{Fore.CYAN}{Style.BRIGHT}{'#' * 80}")
    print("###                                PYTHON FAQ CHATBOT                               ###")
    print("### Hybrid Mode: FAQ Match (Retrieval) OR Gemini AI (Generative)                  ###")
    print(f"{'#' * 80}{Style.RESET_ALL}")
    print(f"\n{Fore.YELLOW}>>> Ask a question about Python programming. Type 'quit' or 'exit' to end. <<<{Style.RESET_ALL}\n")

    while True:
        try:
            user_input = input(f"{Fore.GREEN}YOU >> {Style.RESET_ALL}")
        except EOFError:
            user_input = 'quit'

        if user_input.lower() in ['quit', 'exit']:
            print(format_bot_response("Goodbye! Thank you for using the Python FAQ Chatbot.", 1.0, "SYSTEM"))
            break

        answer, similarity, mode = hybrid_chatbot_response(user_input)
        print(format_bot_response(answer, similarity, mode))

if __name__ == '__main__':
    start_chat()


################################################################################
###                                PYTHON FAQ CHATBOT                               ###
### Hybrid Mode: FAQ Match (Retrieval) OR Gemini AI (Generative)                  ###
################################################################################

>>> Ask a question about Python programming. Type 'quit' or 'exit' to end. <<<

YOU >> What is a decorator in Python and how do you use it?

╔════════════════════════════════════════════════════════════════════════════════╗
║  🤖 BOT Response (Gemini AI):                                           ║
╠────────────────────────────────────────────────────────────────────────────────╣
║ A decorator in Python is a design pattern that allows you to modify or
║    extend the functionality of a function or class without permanently
║    altering its source code. It's essentially a function that takes another
║    function as an argument, wraps it inside an inner f